# Lección 8.3 — Clustering jerárquico y DBSCAN: árboles genealógicos y multitudes

> **Módulo 8 · Aprendizaje no supervisado** · Nivel 3 — Practicante
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 75 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 8.2 (K-means, silueta)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar cómo el **agrupamiento jerárquico** construye grupos uniendo los más parecidos, y leer un **dendrograma**.
- Explicar cómo **DBSCAN** encuentra grupos como "zonas densas" y marca el **ruido**.
- Predecir el efecto de `linkage`, `eps` y `min_samples`.
- Elegir entre K-means, jerárquico y DBSCAN según la forma de los datos.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.datasets import make_blobs, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, HDBSCAN
from sklearn.neighbors import NearestNeighbors

plt.rcParams["figure.figsize"] = (8, 4.5)

def dibujar(ax, X, etiquetas, titulo):
    # El ruido (etiqueta -1) se dibuja como una x gris
    ruido = etiquetas == -1
    ax.scatter(*X[~ruido].T, c=etiquetas[~ruido], cmap="tab10", s=12, vmin=0, vmax=9)
    ax.scatter(*X[ruido].T, c="lightgray", marker="x", s=15)
    ax.set_title(titulo, fontsize=10); ax.set_xticks([]); ax.set_yticks([])

print("✅ Todo listo")

---
# Parte A — Agrupamiento jerárquico

## 1. 📖 La intuición: el árbol genealógico de los datos 🌳

Imagina una fiesta donde nadie se conoce. Cada persona empieza **sola**. Luego, una y otra vez:
1. Las **dos** personas (o grupitos) que más se parecen se juntan.
2. Se repite hasta que todos forman **un solo grupo**.

Si anotas **el orden** en que se fueron juntando y **qué tan distintos** eran al unirse, obtienes un árbol, como un árbol genealógico. Se llama **dendrograma**.

Lo mejor: no tienes que decidir `k` al principio. Primero construyes el árbol completo y **después** decides dónde "cortarlo".

### ✋ Ejercicio 1 — Agrupa a mano
Cinco personas están en una calle, en estas posiciones (metros):

```
A=1  B=2            C=6  D=7                       E=15
```

Regla de unión: la distancia entre dos grupos es la de sus **miembros más cercanos**.

1. ¿En qué orden se van uniendo? Anota la distancia de cada unión.
2. Si "cortas" el árbol a una altura de **5** metros (no se permiten uniones de más de 5 m), ¿cuántos grupos quedan?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Uniones:
   - **A+B** a distancia 1 y **C+D** a distancia 1 (empatan).
   - **{A,B} + {C,D}** a distancia 4 (lo más cercano: B=2 y C=6).
   - **{A,B,C,D} + E** a distancia 8 (D=7 y E=15).
2. Con corte en 5, la última unión (8) no se permite: quedan **2 grupos**, {A, B, C, D} y {E}. Si cortaras en 2, quedarían **3**: {A,B}, {C,D} y {E}.

Compruébalo en el dendrograma: la altura de cada "puente" es la distancia de la unión.
</details>

In [ ]:
calle = np.array([[1], [2], [6], [7], [15]])
uniones = linkage(calle, method="single")      # "single": distancia entre los miembros más cercanos

dendrogram(uniones, labels=["A", "B", "C", "D", "E"])
plt.axhline(5, color="red", ls="--", label="corte en 5")
plt.ylabel("Distancia de la unión"); plt.legend(); plt.show()

> 📝 Para leer un dendrograma: traza una **línea horizontal** a la altura del corte y **cuenta cuántas líneas verticales cruza**. Ese es el número de grupos.

---
## 2. 📖 El concepto: ¿qué distancia hay entre dos **grupos**?

Entre dos puntos es fácil. Pero ¿entre dos grupos? Hay varias respuestas, y se llaman **enlaces** (*linkage*):

| Enlace | Distancia entre dos grupos = | Personalidad |
|---|---|---|
| `single` (simple) | La de sus miembros **más cercanos** | Sigue "cadenas": bueno para formas alargadas, pero un puente de ruido une todo |
| `complete` (completo) | La de sus miembros **más lejanos** | Grupos compactos y de tamaño parecido |
| `average` (promedio) | El promedio de todas las distancias | Un término medio |
| `ward` | Cuánto **aumenta la inercia** al unirlos (como K-means) | El más usado; grupos redondos y equilibrados. **Por defecto en scikit-learn** |

Veamos un ejemplo con significado: 10 animales descritos por 5 características.

In [ ]:
animales = pd.DataFrame({
    #            patas  vuela  nada  pelo  peso_kg(log10)
    "perro":    [4, 0, 0, 1, 1.3],
    "lobo":     [4, 0, 0, 1, 1.6],
    "gato":     [4, 0, 0, 1, 0.6],
    "águila":   [2, 1, 0, 0, 0.7],
    "gorrión":  [2, 1, 0, 0, -1.6],
    "pingüino": [2, 0, 1, 0, 1.4],
    "pato":     [2, 1, 1, 0, 0.1],
    "delfín":   [0, 0, 1, 0, 2.2],
    "tiburón":  [0, 0, 1, 0, 2.9],
    "salmón":   [0, 0, 1, 0, 0.6],
}, index=["patas", "vuela", "nada", "pelo", "peso"]).T
animales

### 🔮 Ejercicio 2
Antes de ejecutar: ¿cuáles crees que serán las **primeras dos parejas** en unirse? Si cortas el árbol en **3 grupos**, ¿cuáles esperas?

✍️ *Mi apuesta:*

In [ ]:
X_anim = StandardScaler().fit_transform(animales)
dendrogram(linkage(X_anim, method="ward"), labels=animales.index.tolist())
plt.ylabel("Distancia de la unión (ward)"); plt.xticks(rotation=45); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

Las primeras uniones (los "puentes" más bajos) son **perro + lobo**, enseguida **delfín + tiburón**, y luego el **gato** se suma a los caninos. Más arriba se unen **pingüino + salmón** y **águila + gorrión**, y el **pato** se suma a estos dos.

Arriba del todo, el último corte separa a los **mamíferos terrestres** (gato, perro, lobo) de todos los demás. Cortando en **3 grupos**: terrestres · nadadores (delfín, tiburón, pingüino, salmón) · aves voladoras (pato, águila, gorrión).

¿Te sorprende el pingüino junto al salmón? Al algoritmo **solo le dimos 5 columnas**: el pingüino no vuela, nada y pesa algo parecido a un salmón grande. No sabe nada de biología. **Los grupos reflejan las columnas que eliges**, no "la verdad" (Lección 8.1).

El dendrograma es un **mapa completo** de parecidos: puedes elegir el nivel de detalle que necesites sin volver a entrenar.
</details>

### 📖 Cortar el árbol con scikit-learn
`AgglomerativeClustering` hace todo esto y te devuelve los grupos. Le puedes decir:
- `n_clusters=4` → "córtalo donde queden 4 grupos", **o**
- `distance_threshold=3` (con `n_clusters=None`) → "córtalo a esta altura".

In [ ]:
N_GRUPOS = 4     # ✏️ prueba 2, 3, 6
agrupador = AgglomerativeClustering(n_clusters=N_GRUPOS, linkage="ward")
animales["grupo"] = agrupador.fit_predict(X_anim)
animales.sort_values("grupo")[["grupo"]].T

### ✏️ Ejercicio 3
Prueba `N_GRUPOS = 2` y `N_GRUPOS = 6`. Cuando pasas de 4 a 6 grupos, ¿los grupos anteriores se **reorganizan por completo** o simplemente se **parten**?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Simplemente **se parten**: los grupos de 6 son subdivisiones de los de 4, y los de 4 son subdivisiones de los de 2. Es la ventaja de la **jerarquía**: los grupos son coherentes a cualquier nivel (como país → estado → ciudad).

Con K-means esto **no** está garantizado: con k = 4 y k = 6 podrías obtener grupos que se cruzan de forma extraña.
</details>

---
## 3. 📖 ¿Qué pasa si… cambias el enlace?

### 🔮 Ejercicio 4
Volvemos a las dos lunas (donde K-means fracasó en la Lección 8.2). Probaremos el enlace `ward` y el enlace `single`, ambos con 2 grupos. ¿Cuál separará bien las lunas? Pista: piensa en la "personalidad" de cada uno en la tabla.

✍️ *Mi apuesta:*

In [ ]:
X_lunas, _ = make_moons(n_samples=300, noise=0.05, random_state=0)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, enlace in zip(axes, ["ward", "single"]):
    etiquetas = AgglomerativeClustering(n_clusters=2, linkage=enlace).fit_predict(X_lunas)
    dibujar(ax, X_lunas, etiquetas, f"linkage = '{enlace}'")
plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- **`ward`** se comporta como K-means: busca grupos redondos y corta las lunas mal.
- **`single`** las separa **perfectamente**: va uniendo cada punto con su vecino más cercano, "siguiendo la cadena" a lo largo de cada luna.

Pero `single` tiene un talón de Aquiles: si hubiera unos pocos puntos de ruido formando un **puente** entre las lunas, las uniría en un solo grupo. En el Paso 3 de los ejercicios finales lo verás. Para formas raras **con ruido** existe algo mejor…
</details>

---
# Parte B — DBSCAN

## 4. 📖 La intuición: las multitudes en una plaza 🧑‍🤝‍🧑

Mira una plaza desde un edificio alto. Ves **multitudes** (alrededor de un músico, en la fila de un puesto de tacos…) y personas **sueltas** caminando. Nadie te dijo cuántas multitudes hay; las reconoces porque ahí **la gente está apretada**.

DBSCAN (*Density-Based Spatial Clustering of Applications with Noise*) hace lo mismo con dos reglas:
- **`eps`**: el "radio del brazo": qué tan cerca tiene que estar alguien para contar como **vecino**.
- **`min_samples`**: cuántas personas (contándote a ti) tiene que haber dentro de tu radio para que estés **en una multitud**.

Con eso, cada punto es de uno de tres tipos:

| Tipo | Regla | En la plaza |
|---|---|---|
| **Núcleo** (*core*) | Tiene al menos `min_samples` puntos a distancia ≤ `eps` (contándose) | Alguien en medio de la multitud |
| **Borde** | No es núcleo, pero está a ≤ `eps` de un núcleo | Alguien en la orilla de la multitud |
| **Ruido** | Ni núcleo ni borde | Un paseante solitario |

Los núcleos que se "tocan" forman un mismo grupo. **El ruido no pertenece a ningún grupo** (scikit-learn le pone la etiqueta **−1**).

### ✋ Ejercicio 5 — DBSCAN a mano
Ocho personas en una calle, en las posiciones:

```
1  2  3  4        10            20  21  22
```

Usa `eps = 1.5` y `min_samples = 3`.
1. ¿Quiénes son **núcleo**? (Cuenta cuántos hay a distancia ≤ 1.5 de cada uno, incluyéndolo a él.)
2. ¿Quiénes son **borde** y quiénes **ruido**?
3. ¿Cuántos grupos hay?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Núcleos: 2, 3 y 21.** Por ejemplo, el 2 tiene a 1, 2 y 3 dentro de su radio (3 personas). El 1 solo tiene a 1 y 2 (2 personas): no alcanza.
2. **Bordes: 1, 4, 20 y 22** (no son núcleo, pero están a ≤ 1.5 de uno). **Ruido: 10**, que no tiene a nadie cerca.
3. **2 grupos**: {1, 2, 3, 4} y {20, 21, 22}. El 10 queda fuera, como ruido.

Compruébalo:
</details>

In [ ]:
calle2 = np.array([1, 2, 3, 4, 10, 20, 21, 22]).reshape(-1, 1)
db = DBSCAN(eps=1.5, min_samples=3).fit(calle2)

tipo = np.where(db.labels_ == -1, "ruido", "borde").astype(object)
tipo[db.core_sample_indices_] = "núcleo"
pd.DataFrame({"posición": calle2.ravel(), "grupo": db.labels_, "tipo": tipo})

---
## 5. 📖 DBSCAN en acción: formas raras y ruido

Lo que hace especial a DBSCAN:
- **No le dices cuántos grupos hay**: los descubre.
- Encuentra grupos de **cualquier forma** (sigue la densidad, como `single`).
- **Marca el ruido** en vez de forzarlo a un grupo.

### 🔮 Ejercicio 6
Tomamos las lunas y les agregamos 30 puntos de ruido esparcidos por todas partes. Compararemos K-means (k = 2) con DBSCAN. ¿Qué hará cada uno con los puntos de ruido?

✍️ *Mi apuesta:*

In [ ]:
rng = np.random.default_rng(0)
ruido = rng.uniform([-1.5, -1], [2.5, 1.5], size=(30, 2))
X_ruido = np.vstack([X_lunas, ruido])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
dibujar(axes[0], X_ruido, KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(X_ruido), "K-means (k=2)")
etiq_db = DBSCAN(eps=0.15, min_samples=5).fit_predict(X_ruido)
dibujar(axes[1], X_ruido, etiq_db, "DBSCAN (eps=0.15, min_samples=5)")
plt.show()
print("Grupos encontrados por DBSCAN:", len(set(etiq_db)) - (-1 in etiq_db))
print("Puntos marcados como ruido   :", (etiq_db == -1).sum())

<details><summary><b>👀 Ver respuesta</b></summary>

- **K-means** corta las lunas mal y **mete cada punto de ruido** en algún grupo: no tiene forma de decir "este no es de nadie".
- **DBSCAN** encuentra las **2 lunas** sin que le digas cuántas hay y marca como ruido (✖️ gris) a **22** puntos, todos de los 30 que agregamos. Algunos puntos de ruido que cayeron justo encima de una luna se quedan en ella, lo cual es razonable.

Esto hace de DBSCAN un primo cercano de la **detección de anomalías** (Lección 8.6).
</details>

---
## 6. 📖 Hiperparámetros: `eps` y `min_samples`

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `eps` | Radio del vecindario | Menos ruido; los grupos se **fusionan** (al extremo, todo es un grupo) | Más ruido; los grupos se **rompen** (al extremo, todo es ruido) | El "codo" de la gráfica de k-distancias |
| `min_samples` | Cuántos vecinos hacen falta para ser núcleo | Exige más densidad: más ruido, grupos más "serios" | Grupos con pocos puntos; menos ruido | ≈ 2 × número de columnas (mínimo 4–5) |
| `metric` | Cómo se mide la distancia | — | — | `"euclidean"` (¡con datos escalados!) |

### 🔮 Ejercicio 7 — El radio del brazo
Vamos a usar `eps` = **0.05**, **0.15** y **0.8** con las lunas con ruido. Para cada uno, predice: ¿muchos grupos, 2 grupos o 1 grupo? ¿Mucho ruido o poco?

✍️ *Mis predicciones:*

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, eps in zip(axes, [0.05, 0.15, 0.8]):     # ✏️ prueba otros valores
    e = DBSCAN(eps=eps, min_samples=5).fit_predict(X_ruido)
    n_grupos = len(set(e)) - (-1 in e)
    dibujar(ax, X_ruido, e, f"eps={eps}: {n_grupos} grupos, {(e == -1).sum()} de ruido")
plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- **`eps=0.05`**: el brazo es tan corto que casi nadie tiene 5 vecinos: **casi todo es ruido** y quedan muchos grupitos sueltos.
- **`eps=0.15`**: el punto justo: **2 grupos** y el ruido bien detectado.
- **`eps=0.8`**: el brazo es tan largo que todo toca con todo: **1 solo grupo**, sin ruido.

Es la curva en "U" de siempre (Lección 4.4), con otra perilla: muy bajo "rompe" los grupos, muy alto los "funde".
</details>

### 📖 ¿Cómo elijo `eps` sin adivinar? La gráfica de k-distancias
Para cada punto, medimos la distancia a su **vecino número `min_samples`** y las ordenamos de menor a mayor. Los puntos dentro de una multitud tienen esa distancia **pequeña**; los puntos de ruido, **grande**. Donde la curva se dispara (el "codo") está un buen `eps`.

In [ ]:
MIN_SAMPLES = 5
vecinos = NearestNeighbors(n_neighbors=MIN_SAMPLES).fit(X_ruido)
distancias, _ = vecinos.kneighbors(X_ruido)
k_dist = np.sort(distancias[:, -1])        # distancia al vecino número MIN_SAMPLES (contándose a sí mismo)

plt.plot(k_dist)
plt.axhline(0.15, color="red", ls="--", label="eps = 0.15")
plt.ylabel(f"Distancia al vecino n.º {MIN_SAMPLES}"); plt.xlabel("Puntos (ordenados)")
plt.legend(); plt.grid(alpha=.3); plt.show()

### ✋ Ejercicio 8
Mirando la gráfica: ¿aproximadamente cuántos puntos están **por encima** de la línea roja? ¿Qué serán para DBSCAN?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Unas pocas decenas de puntos (los del extremo derecho, donde la curva se dispara) quedan por encima de 0.15. No pueden ser núcleo: serán **borde** si tienen un núcleo cerca o, la mayoría, **ruido**. Es coherente con los 22 puntos de ruido del Ejercicio 6. El codo de la curva (≈0.15–0.2) es el rango razonable para `eps`.
</details>

### 📖 La debilidad de DBSCAN: densidades distintas
Un solo `eps` para todo el mapa supone que todas las multitudes son **igual de apretadas**. ¿Y si hay un grupo muy denso y otro muy disperso?

### 🔮 Ejercicio 9
Tres grupos: dos muy apretados y cercanos entre sí, y uno disperso. ¿Podrá DBSCAN encontrar los 3 con un solo `eps`? Compáralo con **HDBSCAN**, una versión moderna que **prueba todos los `eps` a la vez** y se queda con los grupos más estables (solo le dices el tamaño mínimo de un grupo).

✍️ *Mi apuesta:*

In [ ]:
X_dens, _ = make_blobs(n_samples=[200, 200, 200], centers=[[0, 0], [1.8, 0], [8, 4]],
                       cluster_std=[0.2, 0.2, 1.5], random_state=0)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, eps in zip(axes[:2], [0.3, 1.0]):
    e = DBSCAN(eps=eps, min_samples=5).fit_predict(X_dens)
    dibujar(ax, X_dens, e, f"DBSCAN eps={eps}: {len(set(e)) - (-1 in e)} grupos, {(e == -1).sum()} ruido")
e = HDBSCAN(min_cluster_size=20, copy=True).fit_predict(X_dens)
dibujar(axes[2], X_dens, e, f"HDBSCAN: {len(set(e)) - (-1 in e)} grupos, {(e == -1).sum()} ruido")
plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- **`eps=0.3`** (pensado para los grupos densos): encuentra bien los dos apretados, pero el disperso se deshace en **ruido** (128 puntos) y grupitos: 8 "grupos" en total.
- **`eps=1.0`** (pensado para el disperso): este sí aparece, pero los dos grupos densos, que están cerca entre sí, se **fusionan** en uno.
- **HDBSCAN** encuentra los **3 grupos** correctos y marca como ruido solo **2** puntos sueltos.

Por eso hoy, cuando dudas, **HDBSCAN** (`sklearn.cluster.HDBSCAN`) suele ser mejor punto de partida que DBSCAN: su perilla principal (`min_cluster_size`) es mucho más fácil de elegir que `eps`.
</details>

---
## 7. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| DBSCAN marca **todo** como ruido | `eps` demasiado pequeño o `min_samples` muy alto | Subir `eps` (usa la gráfica de k-distancias) |
| DBSCAN devuelve **un solo** grupo gigante | `eps` demasiado grande | Bajar `eps` |
| `eps` que funcionaba deja de funcionar al añadir una columna | Datos **sin escalar** o más dimensiones (las distancias crecen) | Escalar; recalcular la gráfica de k-distancias |
| Un grupo denso se fusiona con otro, o un grupo disperso desaparece | Densidades distintas: un solo `eps` no sirve | HDBSCAN |
| El jerárquico tarda muchísimo o se queda sin memoria | Calcula **todas** las distancias entre pares: crece con n² | Usarlo con ≤ ~10,000 puntos, o agrupar antes con K-means |
| `linkage="single"` une todo en una "cadena" | Puentes de ruido entre grupos | `ward`/`average`, o DBSCAN/HDBSCAN |
| Necesitas asignar clientes **nuevos** y DBSCAN no tiene `predict` | DBSCAN y el jerárquico no están pensados para datos nuevos | Reentrenar, o entrenar un clasificador (p. ej. KNN) sobre los grupos encontrados |

---
## 8. 📋 Fichas de los modelos

| | Agrupamiento jerárquico (aglomerativo) | DBSCAN / HDBSCAN |
|---|---|---|
| **Intuición** | Árbol genealógico: unir siempre a los dos más parecidos | Multitudes en una plaza: grupos = zonas densas |
| **Cómo funciona** | Une grupos paso a paso según el **enlace**; luego cortas el dendrograma | Expande grupos desde los puntos **núcleo**; lo demás es ruido |
| **Hiperparámetros clave** | `n_clusters` o `distance_threshold`, `linkage` | `eps`, `min_samples` (HDBSCAN: `min_cluster_size`) |
| **¿Hay que elegir k?** | Sí, pero **después** de ver el dendrograma | **No**: lo descubre |
| **✅ Fortalezas** | Dendrograma = mapa completo de parecidos · Grupos coherentes a cualquier nivel | Formas arbitrarias · Detecta **ruido** · No necesita k |
| **❌ Debilidades** | Lento con muchos datos (n²) · No asigna datos nuevos | Sensible a `eps` y a la escala · Le cuestan las densidades distintas (DBSCAN) · No asigna datos nuevos |
| **Úsalo cuando…** | Tienes cientos o pocos miles de datos y quieres ver la estructura a varios niveles (taxonomías, genes, documentos) | Hay formas raras, ruido o atípicos, o no sabes cuántos grupos hay (datos geográficos, sensores) |

### 🗺️ ¿Cuál uso?

| Si tus datos… | Prueba primero |
|---|---|
| Son grupos redondos, muchos datos y necesitas asignar nuevos | **K-means** (8.2) |
| Son pocos y quieres ver la jerarquía / elegir k viendo el árbol | **Jerárquico** (`ward`) |
| Tienen formas raras, ruido o no sabes cuántos grupos hay | **HDBSCAN** / DBSCAN |

## 9. 🧠 Resumen en 5 puntos
1. El **jerárquico** une siempre a los dos grupos más parecidos y registra todo en un **dendrograma**; lo cortas a la altura que quieras.
2. El **enlace** define la distancia entre grupos: `ward` busca grupos redondos; `single` sigue cadenas (y sufre con el ruido).
3. **DBSCAN** define grupos como **zonas densas**: puntos núcleo, borde y **ruido** (−1). No necesita k.
4. `eps` bajo rompe los grupos y llena todo de ruido; `eps` alto los funde. Elígelo con la gráfica de **k-distancias**.
5. Con densidades distintas, **HDBSCAN** es más robusto y más fácil de ajustar (`min_cluster_size`).

---
## 10. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈30 min).

### 10.1 Repaso rápido (opción múltiple)

**1.** En un dendrograma, trazas una línea horizontal y cruza 5 líneas verticales. ¿Qué significa?
- a) Hay 5 atípicos · b) Cortando a esa altura obtienes 5 grupos · c) El árbol tiene 5 niveles · d) k debe ser 5 siempre

**2.** ¿Qué etiqueta pone scikit-learn a los puntos de ruido de DBSCAN?
- a) 0 · b) `None` · c) −1 · d) El grupo más cercano

**3.** Con DBSCAN, subes `eps` muchísimo. ¿Qué esperas?
- a) Todo ruido · b) Un solo grupo · c) Más grupos · d) Nada cambia

**4.** ¿Cuál de estos métodos **no** necesita que le digas el número de grupos?
- a) K-means · b) Jerárquico con `n_clusters=3` · c) DBSCAN · d) Ninguno

**5.** Tienes 2 millones de clientes. ¿Qué método es el **menos** práctico?
- a) K-means · b) MiniBatchKMeans · c) Jerárquico aglomerativo · d) Todos iguales

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Cada línea vertical cruzada es una rama = un grupo.
2. **c)** −1 = "no pertenece a ningún grupo".
3. **b)** Si el radio abarca todo, todos los puntos son vecinos: un solo grupo.
4. **c)** DBSCAN descubre el número de grupos (también HDBSCAN, y el jerárquico si usas `distance_threshold` en vez de `n_clusters`).
5. **c)** Necesita las distancias entre **todos los pares**: con 2 millones son 2 billones de pares.
</details>

### 10.2 Preguntas para pensar (sin ejecutar)

**A.** Una bióloga quiere agrupar 300 especies de plantas por sus características y dice: "No sé si quiero 5 familias o 20 subfamilias; quiero ver ambas cosas". ¿Qué método le recomiendas y por qué?

**B. Caso de análisis 💼** Una app de reparto tiene las coordenadas GPS de 50,000 pedidos de la última semana. Quiere encontrar las "zonas calientes" para ubicar repartidores en espera. Un analista propone K-means con k = 10. ¿Qué problemas le ves? ¿Qué propondrías?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** **Agrupamiento jerárquico**: con 300 datos es rápido, y un único dendrograma le muestra **todos** los niveles. Cortando arriba obtiene ~5 familias y cortando abajo ~20 subfamilias, y además las subfamilias quedan **dentro** de las familias, lo que tiene sentido biológico.

**B.** Problemas de K-means aquí: (1) hay que adivinar k = 10; (2) las zonas calientes tienen formas irregulares (siguen avenidas, colonias); (3) **todo** pedido queda asignado a una zona, incluidos los pedidos aislados en las afueras, que arrastrarían los centros. Propuesta: **DBSCAN/HDBSCAN** sobre las coordenadas: encuentra zonas densas de cualquier forma, descubre cuántas hay y deja los pedidos aislados como ruido. (Ojo: con coordenadas geográficas conviene usar una distancia adecuada, como `metric="haversine"`.)
</details>

### 10.3 🧩 Práctica integradora: "Puntos calientes de una ciudad"

El área de movilidad de una ciudad registró dónde se piden más **bicicletas compartidas** (coordenadas en km respecto al centro). Quiere saber **cuántas zonas de alta demanda** hay y dónde poner nuevas estaciones. Además hay pedidos dispersos por toda la ciudad que **no** forman zona.

Lo harás en 5 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Mira los datos**

In [ ]:
rng = np.random.default_rng(7)
zona_centro  = rng.normal([0, 0], 0.35, (250, 2))                                 # plaza central
t = rng.uniform(0, np.pi, 200)
zona_avenida = np.c_[3 + 2.2 * np.cos(t), 1 + 2.2 * np.sin(t)] + rng.normal(0, 0.12, (200, 2))  # avenida curva
zona_univ    = rng.normal([-3, 3], 0.3, (120, 2))                                 # universidad
dispersos    = rng.uniform([-5, -3], [7, 5], (80, 2))                             # pedidos sueltos
pedidos = np.vstack([zona_centro, zona_avenida, zona_univ, dispersos])

plt.scatter(*pedidos.T, s=8)
plt.xlabel("km este–oeste"); plt.ylabel("km norte–sur"); plt.title("Pedidos de bicicleta"); plt.show()

✋ A simple vista, ¿cuántas zonas de alta demanda ves? ¿Qué forma tienen?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**3 zonas**: una mancha redonda en el centro, otra más pequeña arriba a la izquierda (universidad) y una **curva** en forma de arco (la avenida). Además, ~80 pedidos sueltos por toda la ciudad. Una forma curva + ruido: mal terreno para K-means.

> 📝 Aquí **no escalamos**: las dos columnas ya están en la misma unidad (km), y escalar deformaría las distancias reales.
</details>

**Paso 2 🔮 — K-means como primer intento.** Con k = 3, ¿qué crees que pasará con el arco y con los pedidos sueltos?

✍️ *Mi apuesta:*

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
dibujar(ax, pedidos, KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(pedidos), "K-means (k=3)")
plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

K-means **asigna todos** los pedidos sueltos a alguna zona (las zonas "se estiran" hasta las afueras) y la frontera entre grupos es recta, así que el arco puede quedar mezclado con los pedidos dispersos de alrededor. Para el negocio es engañoso: pondría estaciones pensando que hay demanda donde solo hay pedidos aislados.
</details>

**Paso 3 🔮 — El enlace `single` y el ruido.** En la Parte A, `single` separó las lunas perfectamente. ¿Funcionará aquí, con 80 pedidos sueltos entre las zonas?

✍️ *Mi apuesta:*

In [ ]:
e_single = AgglomerativeClustering(n_clusters=3, linkage="single").fit_predict(pedidos)
print("Tamaño de cada grupo:", np.bincount(e_single))

<details><summary><b>👀 Ver respuesta</b></summary>

**No.** Un grupo se queda con **645** de los 650 pedidos y los otros dos tienen solo **3 y 2** pedidos. Los pedidos sueltos forman "puentes" que encadenan las tres zonas, y lo único "lejano" que queda son unos pocos pedidos aislados en las afueras. Es el talón de Aquiles del enlace `single`.
</details>

**Paso 4 ✏️ — DBSCAN.** Primero mira la gráfica de k-distancias y elige un `eps`. Escríbelo en `EPS` y ejecuta. Intenta que salgan **3 zonas** y que el ruido se parezca a los ~80 pedidos sueltos.

In [ ]:
MIN_SAMPLES = 8    # ✏️
dist, _ = NearestNeighbors(n_neighbors=MIN_SAMPLES).fit(pedidos).kneighbors(pedidos)
plt.plot(np.sort(dist[:, -1])); plt.grid(alpha=.3)
plt.ylabel(f"Distancia al vecino n.º {MIN_SAMPLES}"); plt.xlabel("Pedidos (ordenados)"); plt.show()

In [ ]:
EPS = 0.3          # ✏️ elige mirando el codo de la gráfica de arriba
e_db = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES).fit_predict(pedidos)

fig, ax = plt.subplots(figsize=(6, 4.5))
dibujar(ax, pedidos, e_db, f"DBSCAN eps={EPS}")
plt.show()
print("Zonas encontradas:", len(set(e_db)) - (-1 in e_db))
print("Pedidos de ruido :", (e_db == -1).sum())
print("Pedidos por zona :", np.bincount(e_db[e_db >= 0]))

<details><summary><b>👀 Ver respuesta</b></summary>

El codo de la gráfica está alrededor de **0.3–0.4**. Con `EPS = 0.3` DBSCAN encuentra **3 zonas** (centro ≈250, avenida ≈200, universidad ≈120 pedidos, más algún pedido suelto que cayó dentro) y marca como ruido a **77** pedidos (casi todos los ~80 sueltos). Prueba `EPS = 0.1` (todo se rompe en ruido) y `EPS = 1.0` (las zonas absorben a casi todos los pedidos sueltos) para ver los extremos.
</details>

**Paso 5 ✋ — El informe.** Escribe dos frases para el área de movilidad: ¿cuántas estaciones nuevas recomendarías y dónde? ¿Qué harías con los pedidos de "ruido"?

✍️ *Mi informe:*

<details><summary><b>👀 Ver respuesta</b></summary>

*"Hay 3 zonas de alta demanda: la plaza central (la más grande, ~250 pedidos por semana), el corredor de la avenida (~200, repartidos a lo largo de un arco, así que conviene poner **varias** estaciones pequeñas a lo largo de la avenida en vez de una sola en su centro) y la universidad (~120). Los ~75 pedidos dispersos no justifican estaciones fijas; se pueden atender con bicis sin estación o revisarse dentro de unos meses por si alguno empieza a formar una zona."*

Fíjate en el detalle del arco: el **centro** de esa zona (lo que daría K-means) cae en medio de la curva, ¡donde casi no hay pedidos!
</details>

---
## 📚 Para profundizar
- 🎮 **Interactivo:** *Visualizing DBSCAN Clustering* (Naftali Harris): mueve `eps` y `min_samples` y mira cómo se forman los grupos.
- 📺 **Video:** StatQuest, "Hierarchical Clustering" y "Clustering with DBSCAN, Clearly Explained" (inglés, con subtítulos).
- 📘 **Documentación:** scikit-learn, guía de usuario, sección *Clustering*: la gráfica comparativa de todos los algoritmos sobre distintas formas es oro puro.

---
⬅️ **Anterior:** [8.2 K-means: repartir carritos de helado en la playa](8.2-k-means.ipynb) · ➡️ **Siguiente:** [8.4 Reducción de dimensionalidad: PCA, la mejor "foto" de los datos](8.4-pca.ipynb)